# S5 · RLVR: verifier design and reasoning tasks

Study path stage 5 of 7 · Chapter 13 · Day 23 · **DGX Spark GPU first**

In S4 the reward came from a learned reward model `rm`. **RLVR** (reinforcement learning with
verifiable rewards) keeps the same update rule, GRPO here, and replaces `rm` with a **verifier**: a
program that compares the policy's answer with a per-prompt ground truth and returns 0 or 1. So RLVR
is a choice of *reward source*, not a new update rule (D32), and it needs a reference answer for every
prompt (D31). After this notebook you'll be able to write a verifier and know its false positives and
false negatives, generate verifiable tasks with a difficulty knob, estimate pass@k without bias, say
which prompts give GRPO any gradient, and run and honestly evaluate a short GRPO + verifier run on
Qwen3-0.6B.

| Part | Question | Discussions in the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
|---|---|---|
| A · Extraction | Which part of a response is "the answer"? | D31, D32 |
| B · Verifier | When are two answers equal, and what does a verifier reward wrongly? | D32, D33 |
| C · Tasks and estimators | How do difficulty, pass@k and group size decide whether a 0/1 reward carries signal? | D33 |
| D · Toy GRPO + verifier | What happens to prompts that are never or always solved? | D33, D34 |
| E · Qwen3: measure | Which synthetic problems does Qwen3-0.6B solve sometimes? | D33 |
| F · Qwen3: train and evaluate | Does a short GRPO + verifier run change held-out accuracy beyond noise, and is it formatting or arithmetic? | D32–D34 |

RL terms (policy, rollout, reward, advantage, baseline) are defined in the glossary section
[RL terms in the LLM setting](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39) of the learning note.

**How the exercises work.** Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then uses the function in a small
   experiment.

Later cells use your version if its check passed, and the reference otherwise (via
`mine_or_ref(globals(), …)`), so one stuck exercise never blocks the rest. **Predict** cells ask you
to write an answer before running the reference.

**Data.** No dataset is downloaded. The arithmetic word problems are generated in the notebook by a
seeded program (`s5.make_problem`, authored for this course), which also computes each exact
answer. Training and held-out problems come from different seeds and are checked to be disjoint.

**Environment.** The `Python (DGX Spark Native)` kernel with CUDA, and Qwen3-0.6B in the local
Hugging Face cache. Nothing is downloaded. Checks and references come from the tested module
`dongxi_llms.study_path.s5_rlvr`; model plumbing from `dongxi_llms.study_path.qwen`.

**Run time.** Parts A–D take seconds. Parts E–F take about 8–10 minutes on the Spark, almost all of
it sampling from Qwen3.

**Scale and limits.** The toy parts are exact. The Qwen3 run is short (about 25 updates on a few dozen
prompts of one synthetic task family), so it shows the mechanics and measures one small change with
its uncertainty. It doesn't show that RLVR improves general reasoning, and a gain on these problems
can come from output format as much as from arithmetic; Part F measures the split.

In [ ]:
import inspect, math, random, re, sys, time
from fractions import Fraction
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from dongxi_llms.study_path import qwen as Q, s5_rlvr as s5
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
transformers.logging.set_verbosity_error()
random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.family": "DejaVu Sans",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | {torch.cuda.get_device_name()}")

---

# Part A · Extraction: which part of the response is the answer?

A verifier needs three things: the policy's response `y`, the prompt's ground truth `y*`, and a
**contract**, the written rule that says which outputs count as a correct answer. The reward is then

$$r(x, y) = \mathbb{1}\big[\,\text{normalise}(\text{extract}(y)) = y^*(x)\,\big] \in \{0, 1\}.$$

Unlike `rm`, nothing here is learned; the reward is exactly as good as the contract. Each RLVR
prompt therefore carries its own reference answer (D31, D32). The reward-model RL of S4 needed only
prompts.

The first step, **extraction**, picks the answer span out of free text. Our contract: the response
contains **exactly one line** that starts with `Answer:`; the answer is the text after the marker.
Zero such lines (no committed answer) or two (the model hedges) both mean "no answer". We don't take
the last number in the text, because a number in the working is not a commitment.

The pattern below matches one whole line: `^` and `$` anchor at line boundaries (`re.MULTILINE`), and
the group `(.+?)` captures the value without the surrounding spaces.

In [ ]:
print("ANSWER_RE:", s5.ANSWER_RE.pattern, "| flags: MULTILINE")
print(s5.ANSWER_RE.findall("6 × 7 = 42\nAnswer: 42\nAnswer:  24 "))

### Prediction

## Exercise 1 · extract the final answer

Return the captured value when `ANSWER_RE` matches **exactly one** line, otherwise `None`. Two lines.
`re.findall` with one group returns the list of captured strings.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_extract_answer(text):
    found = ...                 # every captured "Answer: …" value, via s5.ANSWER_RE
    return ...                  # the single match, or None when there are 0 or ≥ 2

In [ ]:
ex1_ok = s5.check_extract_answer(globals().get("my_extract_answer"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def extract_answer_ref(text):
    found = s5.ANSWER_RE.findall(text)
    return found[0] if len(found) == 1 else None

s5.check_extract_answer(extract_answer_ref, strict=True)
extract_answer = mine_or_ref(globals(), "my_extract_answer", ex1_ok, extract_answer_ref)
for text in ["Start with 30, add 12.\nAnswer: 42", "Answer: **42**", "So it is 42.\nAnswer: 42\nAnswer: 43"]:
    print(f"{text!r:50} → {extract_answer(text)!r}")

### Reference explanation

### Reference answer and mechanism

`found[0] if len(found) == 1 else None`: one committed answer or nothing. Two equal `Answer:` lines
also return `None`, because the contract says "exactly one line", and a rule with exceptions is
harder to audit than a strict one.

Extraction decides what the policy is graded on, so it also decides what RL teaches. With this rule,
a response that writes `Answer: **42**` passes extraction (the value is `**42**`) but will fail
normalisation in Part B, so the policy is pushed to write plain numbers. A rule that took the last
number in the text would instead reward any response that happens to end on the right number,
including one that hedges between two answers.

---

# Part B · Normalisation, the verifier, and what it rewards wrongly

After extraction we have a string such as `"1,042"`, `"3.50"` or `"-0"`, and the ground truth is an
integer. Comparing strings would reject `"1,042"` against `1042` and `"42.0"` against `42`, which are
**false negatives**: correct answers scored 0. The fix is to convert both sides to an exact number,
then compare numbers.

The contract for a number: an optional sign, digits with optional comma thousands groups
(`1,042`, not `1,00`), an optional decimal part, and at most one sentence-final period. `NUMBER_RE`
encodes it with `[0-9]` rather than `\d`, because Python's `\d` also matches other scripts' digits.
The value is a `Fraction`, which represents `0.1` exactly as 1/10. A `float` can't, so
`float("0.1")` is not equal to `Fraction(1, 10)`, and decimal answers compared as floats can fail.

In [ ]:
print("NUMBER_RE:", s5.NUMBER_RE.pattern)
print(Fraction("3.50"), Fraction("-0"), Fraction("0.1") == 0.1, float("0.1") == Fraction(1, 10))

### Prediction

## Exercise 2 · normalise a number exactly

Write the two `return` lines: `None` when the string doesn't match the contract, otherwise the exact
value as a `Fraction` after removing the thousands commas.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_parse_number(s):
    s = s.strip().removesuffix(".")          # given: one sentence-final period is allowed
    if s5.NUMBER_RE.fullmatch(s) is None:
        return ...                           # outside the contract
    return ...                               # exact value: drop the commas, then Fraction(...)

In [ ]:
ex2_ok = s5.check_parse_number(globals().get("my_parse_number"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def parse_number_ref(s):
    s = s.strip().removesuffix(".")
    if s5.NUMBER_RE.fullmatch(s) is None:
        return None
    return Fraction(s.replace(",", ""))

s5.check_parse_number(parse_number_ref, strict=True)
parse_number = mine_or_ref(globals(), "my_parse_number", ex2_ok, parse_number_ref)
for s in ["1,000", "1000", "3.50", "-0", "42.", "1,00", "**42**", "٤٢"]:
    print(f"{s!r:10} → {parse_number(s)!r}")

### Reference explanation

### Reference answer and mechanism

`Fraction(s.replace(",", ""))` after the regex has accepted `s`. The regex runs first so that the
comma removal can't turn an invalid grouping such as `1,00` into `100`. `Fraction` then makes
`"3.50"`, `"3.5"` and `"7/2"`-valued strings the same number, and `"-0"` equal to `0`.

`"**42**"` and the Arabic-Indic digits `"٤٢"` return `None`: they are outside this contract. That
is a deliberate false negative. Accepting them would need an explicit rule (strip Markdown
emphasis, map digit scripts), and every added rule is one more place where a wrong answer could pass.

Now the reward itself. RLVR rewards are usually **binary**: 1 if the answer is correct, 0
otherwise. Some recipes add a small **format** component α for a well-formed but wrong answer
(DeepSeek-R1 used a separate format reward). With 0 < α < 1 the reward takes three values:

$$r = \begin{cases} 1 & \text{one parseable Answer line, equal to } y^* \\ \alpha & \text{one parseable Answer line, not equal to } y^* \\ 0 & \text{otherwise.} \end{cases}$$

α gives a gradient to a group where every answer is wrong but some are well-formed. It also lets
the policy collect reward without solving the problem, so a run with α > 0 needs its accuracy
reported separately from its reward. This notebook trains with α = 0.

### Prediction

## Exercise 3 · the verifier reward

Use `extract_answer` and `parse_number` (yours if they passed). Write three lines: whether the
response is well-formed, whether it is correct, and the reward. `Fraction == int` compares exactly.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_verifier_reward(text, gold, alpha=0.0):
    answer = extract_answer(text)
    value = None if answer is None else parse_number(answer)
    formatted = ...                 # True when there is one Answer line and it parses
    correct = ...                   # formatted, and the value equals gold
    return ...                      # 1.0 if correct, else alpha for a well-formed answer, else 0

In [ ]:
ex3_ok = s5.check_verifier_reward(globals().get("my_verifier_reward"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def verifier_reward_ref(text, gold, alpha=0.0):
    answer = extract_answer(text)
    value = None if answer is None else parse_number(answer)
    formatted = value is not None
    correct = formatted and value == gold
    return 1.0 if correct else alpha * float(formatted)

s5.check_verifier_reward(verifier_reward_ref, strict=True)
verifier_reward = mine_or_ref(globals(), "my_verifier_reward", ex3_ok, verifier_reward_ref)
print(verifier_reward("Answer: 1,042", 1042), verifier_reward("Answer: 41", 42, alpha=0.1),
      verifier_reward("The answer is 42.", 42, alpha=0.1))

### Reference explanation

### Reference answer and mechanism

`formatted = value is not None`, `correct = formatted and value == gold`, and
`1.0 if correct else alpha * float(formatted)`. A correct answer gets exactly 1, not 1 + α, so the
format bonus never outranks correctness. Everything that reaches the policy goes through this one
function, so its edge cases are part of the task definition (Chapter 13 §13.6).

How much does the contract matter? Below are twelve responses, each with a human label: 1 if the
response states one final value equal to the ground truth, in any format. We score them with three
verifiers:

- **substring**: 1 if the digits of `y*` appear anywhere in the text (a common bug);
- **last number**: 1 if the last number in the text equals `y*` (a common heuristic);
- **strict**: our `verifier_reward`.

A **false positive** is reward 1 on a response labelled 0, and the policy can learn to produce it.
A **false negative** is reward 0 on a response labelled 1, which is a correct answer the policy is
pushed away from.

### Prediction

## Predict 1 · false positives and false negatives of three verifiers

Read the twelve responses printed by the next cell (or `s5.ADVERSARIAL`). For each verifier, how many
false positives and how many false negatives do you expect? Which one would you train with?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
rows = []
for text, gold, label in s5.ADVERSARIAL:
    rows.append({"response": text.replace("\n", " ⏎ "), "gold": gold, "label": label,
                 "substring": s5.substring_reward(text, gold), "last number": s5.last_number_reward(text, gold),
                 "strict": verifier_reward(text, gold)})
adv = pd.DataFrame(rows)
VERIFIERS = ["substring", "last number", "strict"]
summary = pd.DataFrame({v: {"false positives": int(((adv[v] == 1) & (adv.label == 0)).sum()),
                            "false negatives": int(((adv[v] == 0) & (adv.label == 1)).sum())} for v in VERIFIERS})
print(summary.to_string()); print()
fig, ax = plt.subplots(figsize=(10, 5.2))
M = adv[["label"] + VERIFIERS].to_numpy(float)
ax.imshow(M, cmap="Blues", vmin=-0.3, vmax=1.3, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        wrong = j > 0 and M[i, j] != M[i, 0]
        ax.text(j, i, f"{M[i, j]:.0f}" + (" ✗" if wrong else ""), ha="center", va="center",
                color=RED if wrong else "black", fontweight="bold" if wrong else "normal")
ax.set_xticks(range(4), ["human label"] + VERIFIERS)
ax.set_yticks(range(len(adv)), [f"{r[:38]!s}  (y*={g})" for r, g in zip(adv.response, adv.gold)], fontsize=8)
ax.set_title("Reward from three verifiers; ✗ marks a disagreement with the label")
plt.tight_layout(); plt.show()
assert summary.loc["false positives", "strict"] == 0 and summary.loc["false negatives", "strict"] == 3

### Reference explanation

### Reference answer and mechanism

Substring has 7 false positives: every response that mentions 42 anywhere, including `Answer: 420`,
a hedge between two answers and a truncated response. It also has 1 false negative (`1,042`, because
of the comma). Last number has 1 false positive, a check line written after a wrong answer, and 1
false negative (it reads `1,042` as `042`). On this table it looks nearly as good as strict, but its
false positive is easy to produce on purpose: append any line that ends on the right number. Strict
has **no false positives** and 3 false negatives: `**Answer:** 42`, `The answer is 42.` and
`Answer: 42 apples`. All three break the format the prompt asks for.

That is the usual trade-off. A false positive is a behaviour the policy can learn to exploit, so it
does more damage than a false negative. A false negative costs signal on correct answers and pushes
the policy toward the stated format. The strict verifier is the one to train with, provided the
prompt states the format; in Part F we'll measure how much of Qwen3's gain is the format itself.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Rewards from substring, last-number and strict verifiers on twelve labelled responses](../figures/chapter-13/study-path-s5_rlvr-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Tasks with a difficulty knob, pass@k, and when a 0/1 reward carries signal

RLVR needs prompts whose answers a program can compute. We generate them. `s5.make_problem(rng, steps)`
writes a word problem with a start value and `steps` operations (add, give away, boxes of k items,
double or triple, split into bags), and computes the exact answer as it writes each sentence.
`steps` is the **difficulty knob**: more steps mean more chances for an arithmetic slip and longer
responses. The unit tests replay every sentence to confirm the stored answer.

In [ ]:
print(inspect.getsource(s5.make_problem)[:900], "…\n")
for steps in (1, 3, 5):
    p = s5.make_problems(seed=0, steps=steps, n=1)[0]
    print(f"steps={steps} | answer {p['answer']:>4} | {p['question']}")
print("\nInstruction appended to every problem:", s5.INSTRUCTION)

Next, a measurement question. For a prompt with true success rate `p` (the probability that one
sample at temperature 1 is correct), **pass@k** is the probability that at least one of `k`
independent samples is correct: `1 − (1 − p)^k`. We don't know `p`; we have `n` samples of which `c`
are correct. Plugging in `p̂ = c/n` gives `1 − (1 − c/n)^k`, which is **biased**: it's a convex
function of `p̂`, so its average over runs is too low for `k > 1`.

The unbiased estimator (Chen et al., 2021) counts instead: among all `k`-subsets of the `n` samples,
the fraction that contains at least one correct sample.

$$\widehat{\text{pass@}k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

since `C(n−c, k)` subsets contain only incorrect samples. When `n − c < k`, every subset contains a
correct sample and the estimate is 1.

### Prediction

## Exercise 4 · unbiased pass@k

One line with `math.comb`. (`math.comb(a, k)` is 0 when `a < k`, so the edge case needs no branch.)

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pass_at_k(n, c, k):
    return ...                   # 1 − C(n − c, k) / C(n, k)

In [ ]:
ex4_ok = s5.check_pass_at_k(globals().get("my_pass_at_k"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def pass_at_k_ref(n, c, k):
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

s5.check_pass_at_k(pass_at_k_ref, strict=True)
pass_at_k = mine_or_ref(globals(), "my_pass_at_k", ex4_ok, pass_at_k_ref)

# Simulate 20,000 prompts with true p = 0.3, n = 8 samples each, and average both estimators.
g = torch.Generator().manual_seed(0)
P_TRUE, N_S = 0.3, 8
c_sim = (torch.rand(20000, N_S, generator=g) < P_TRUE).sum(1).tolist()
KS = list(range(1, N_S + 1))
bias_table = pd.DataFrame({
    "true": [1 - (1 - P_TRUE) ** k for k in KS],
    "unbiased (mean)": [np.mean([pass_at_k(N_S, c, k) for c in c_sim]) for k in KS],
    "plug-in (mean)": [np.mean([s5.naive_pass_at_k(N_S, c, k) for c in c_sim]) for k in KS]}, index=KS)
print(bias_table.round(4).to_string())

### Reference explanation

### Reference answer and mechanism

`1 − comb(n − c, k) / comb(n, k)`. The table confirms it on 20,000 simulated prompts: the unbiased
estimator's average matches `1 − 0.7^k` to about three decimals for every `k`, while the plug-in
estimate is about 8 points low at `k = 4` (0.683 against 0.760) and the gap grows with `k`. At `k = 1` both are `c/n`.
pass@k needs `n ≥ k` samples per prompt, and its standard error shrinks with the number of prompts,
not with `k`.

pass@k describes evaluation. Training has a related question. GRPO's advantage for sample `i` in a
group of `G` is `A_i = (r_i − mean(r)) / (std(r) + ε)`. With 0/1 rewards, if all `G` samples are
correct, or all are wrong, every `A_i = 0` and the prompt contributes **no gradient** (S4). For
independent samples with success rate `p`, this happens with probability

$$P(\text{zero signal}) = p^G + (1 - p)^G.$$

So a prompt the policy never solves (`p = 0`) and one it always solves (`p = 1`) are both wasted
compute: they cost `G` rollouts and give nothing. That is why RLVR pipelines filter prompts by
measured difficulty (D33).

### Prediction

## Exercise 5 · probability that a group gives no signal

One line; use `**` so that it also works elementwise when `p` is a tensor.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_zero_signal_prob(p, G):
    return ...                   # all G succeed, or all G fail

In [ ]:
ex5_ok = s5.check_zero_signal_prob(globals().get("my_zero_signal_prob"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def zero_signal_prob_ref(p, G):
    return p ** G + (1 - p) ** G

s5.check_zero_signal_prob(zero_signal_prob_ref, strict=True)
zero_signal_prob = mine_or_ref(globals(), "my_zero_signal_prob", ex5_ok, zero_signal_prob_ref)
for p in (0.02, 0.1, 0.5, 0.9):
    print(f"p = {p:4}: " + " | ".join(f"G={G:2d}: {zero_signal_prob(p, G):.3f}" for G in (4, 8, 16)))

### Reference explanation

### Reference answer and mechanism

`p ** G + (1 − p) ** G`. At `p = 0.5` and `G = 8` only 0.8 % of groups are constant. At `p = 0.1`,
43 % are (all eight wrong), and at `p = 0.02` it is 85 %. A larger `G` helps on hard prompts (at
`p = 0.1`, `G = 16` brings it to 19 %) but doubles sampling cost. The useful prompts are those in
the middle: solved sometimes, not never or always.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
ax.plot(KS, bias_table["true"], color=GRAY, lw=4, alpha=.6, label="true 1 − (1 − p)^k")
ax.plot(KS, bias_table["unbiased (mean)"], color=BLUE, marker="o", label="unbiased estimator (mean)")
ax.plot(KS, bias_table["plug-in (mean)"], color=ORANGE, marker="s", label="plug-in 1 − (1 − c/n)^k (mean)")
ax.set(xlabel="k", ylabel="pass@k", title=f"pass@k from n = {N_S} samples, true p = {P_TRUE}")
ax.legend(fontsize=8)
ax = axes[1]
ps = torch.linspace(0, 1, 201)
for G, color in ((4, GRAY), (8, BLUE), (16, ORANGE)):
    ax.plot(ps, zero_signal_prob(ps, G), color=color, label=f"G = {G}")
ax.set(xlabel="success rate p of the prompt", ylabel="P(all G rewards equal)",
       title="Probability that a GRPO group has zero advantage")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Unbiased vs plug-in pass@k, and the zero-signal probability p^G + (1 − p)^G](../figures/chapter-13/study-path-s5_rlvr-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part D · Toy GRPO + verifier on a tabular policy

Before Qwen3, the same loop on a policy small enough to inspect. There are `N = 60` prompts, each
with `K = 10` candidate answers, exactly one correct. The **policy** is a table of logits
`θ ∈ ℝ^{N×K}`, with `π_θ(a | i) = softmax(θ_i)_a`. The verifier is `r = 1[a = a*_i]`. We set the
initial logits so that the probability of the correct answer, `p_i`, is one of six values from
0.0005 to 0.9, ten prompts each. This difficulty is set directly; in Part E we measure it.

Each iteration samples `G = 8` answers for every prompt, scores them with the verifier, computes
group advantages with the given `s5.group_advantages` (S4), and takes one SGD step on the
policy-gradient loss

$$L(\theta) = -\frac{1}{N G} \sum_{i=1}^{N} \sum_{g=1}^{G} A_{ig} \, \log \pi_\theta(a_{ig} \mid i).$$

One step per batch means the PPO ratio is 1 when the gradient is taken, so clipping has no effect
and this plain surrogate has the same gradient as GRPO's clipped one. There is no KL term in the toy.

In [ ]:
print(inspect.getsource(s5.group_advantages))

### Prediction

## Exercise 6 · the policy-gradient loss of a tabular policy

`theta` is `[N, K]`, `actions` is `[N, G]` (long), `adv` is `[N, G]`. Two lines: the log-probability
of each sampled action, then the loss averaged over all `N·G` samples.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_tabular_pg_loss(theta, actions, adv):
    logp = ...                   # [N, G]: log π(a_ig | i), via log_softmax(theta, -1) and .gather(1, actions)
    return ...                   # − mean of adv · logp

In [ ]:
ex6_ok = s5.check_tabular_pg_loss(globals().get("my_tabular_pg_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def tabular_pg_loss_ref(theta, actions, adv):
    logp = torch.log_softmax(theta, -1).gather(1, actions)
    return -(adv * logp).mean()

s5.check_tabular_pg_loss(tabular_pg_loss_ref, strict=True)
tabular_pg_loss = mine_or_ref(globals(), "my_tabular_pg_loss", ex6_ok, tabular_pg_loss_ref)

### Reference explanation

### Reference answer and mechanism

`log_softmax(theta, -1).gather(1, actions)` picks `log π(a_ig | i)`, and `−(adv · logp).mean()` is the
loss. Its gradient for prompt `i` is `−(1/NG) Σ_g A_ig (e_{a_ig} − π_i)`. Two consequences matter
for RLVR:

- On the correct answer's logit, the gradient is `−(1/NG) Σ_{g: correct} A_ig`, since the
  advantages sum to zero. Only correct samples with positive advantage raise it.
- If a group has all-equal rewards, every `A_ig = 0` and the gradient for that row of `θ` is exactly
  zero. With plain SGD the row doesn't change at all.

### Prediction

## Predict 2 · never-solved and always-solved prompts in the toy run

We run 120 iterations at learning rate 20 (applied to a loss averaged over 480 samples). Predict:
(a) the final success probability of the ten prompts that start at `p = 0.0005`, and of those
starting at 0.01; (b) the fraction of groups with zero signal at iteration 1 and at iteration 120.
Will it go down as the policy learns?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
P0 = [0.0005, 0.01, 0.05, 0.2, 0.5, 0.9]
N_EACH, K_ACT, G_TOY, TOY_ITERS, TOY_LR = 10, 10, 8, 120, 20.0
N_TOY = len(P0) * N_EACH
torch.manual_seed(0)
p_init = torch.tensor(P0).repeat_interleave(N_EACH)
correct = torch.randint(0, K_ACT, (N_TOY,))
theta = torch.zeros(N_TOY, K_ACT)
theta[torch.arange(N_TOY), correct] = torch.log((K_ACT - 1) * p_init / (1 - p_init))   # softmax gives p_init
theta0 = theta.clone()
theta.requires_grad_(True)
toy_opt = torch.optim.SGD([theta], lr=TOY_LR)
toy = {"p": [], "zero": [], "zero_pred": []}
ever_solved = torch.zeros(N_TOY, dtype=torch.bool)
for it in range(TOY_ITERS):
    with torch.no_grad():
        pi = torch.softmax(theta, -1)
        p_now = pi[torch.arange(N_TOY), correct]
        actions = torch.multinomial(pi, G_TOY, replacement=True)          # [N, G] rollouts
        R = (actions == correct[:, None]).float()                         # the verifier
        A = s5.group_advantages(R)
    toy["p"].append(p_now.view(len(P0), N_EACH).mean(1))
    toy["zero"].append(float((R.std(1) == 0).float().mean()))
    toy["zero_pred"].append(float(zero_signal_prob(p_now, G_TOY).mean()))
    ever_solved |= R.bool().any(1)
    loss = tabular_pg_loss(theta, actions, A)
    toy_opt.zero_grad(); loss.backward(); toy_opt.step()
toy_p = torch.stack(toy["p"]).numpy()
print(pd.DataFrame({"start p": P0, "final p": toy_p[-1].round(4),
                    "prompts never solved": (~ever_solved).view(len(P0), N_EACH).sum(1).tolist()}).to_string(index=False))
print(f"zero-signal groups: iteration 1 {toy['zero'][0]:.2f} (predicted {toy['zero_pred'][0]:.2f}) | "
      f"iteration {TOY_ITERS} {toy['zero'][-1]:.2f} (predicted {toy['zero_pred'][-1]:.2f})")
never = ~ever_solved
assert torch.equal(theta.detach()[never], theta0[never])       # never solved → row of θ exactly unchanged
print(f"{int(never.sum())} prompts never sampled a correct answer; their logits are bit-for-bit unchanged.")

### Reference explanation

### Reference answer and mechanism

(a) The `p = 0.0005` prompts stay near 0.0005. A prompt that never samples the correct answer gets
an exact zero gradient: the assertion confirms its row of `θ` is bit-for-bit unchanged after 120
updates. Those that sampled it once or twice moved only slightly and fell back into all-wrong groups. The
`p = 0.01` prompts end around 0.2 on average: some got lucky early, rose, and kept rising, while
others stayed at 0.01. Prompts starting at 0.05 or above reach about 0.98–0.99.

(b) About half the groups give no signal at iteration 1 (52 %, against 53 % predicted by
`mean(p^G + (1 − p)^G)`): the hard prompts are all-wrong and the `p = 0.9` prompts are often
all-right. As training succeeds the fraction **rises**, to about 87 % at the end, because solved
prompts give all-correct groups. Zero signal therefore appears at both ends: the hard prompts never
start learning, and the solved prompts stop contributing. Real pipelines refilter or resample the
prompt pool during training for this reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
shades = plt.cm.Blues(np.linspace(0.35, 1.0, len(P0)))
for j, p0 in enumerate(P0):
    ax.plot(toy_p[:, j], color=shades[j], label=f"start p = {p0}")
ax.set(xlabel="iteration", ylabel="mean P(correct) in the bucket", title="Tabular GRPO + verifier, 10 prompts per bucket")
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(toy["zero"], color=BLUE, alpha=.7, label="measured fraction of constant groups")
ax.plot(toy["zero_pred"], color=ORANGE, lw=2, label="predicted: mean of p^G + (1 − p)^G")
ax.set(xlabel="iteration", ylabel="fraction of groups", ylim=(0, 1), title=f"Groups with zero signal (G = {G_TOY})")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Tabular GRPO with a verifier: success per starting difficulty, and the zero-signal fraction](../figures/chapter-13/study-path-s5_rlvr-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part E · Qwen3-0.6B: which problems does it solve sometimes?

Now the real policy. We sample Qwen3-0.6B with thinking off at temperature 1 (the distribution GRPO
samples from), up to `MAX_NEW = 224` new tokens. A response cut off at that limit has no Answer line
and gets reward 0, so the token limit is part of the task. The prompt is the problem followed by
`s5.INSTRUCTION`, which states the answer format.

We draw 12 problems at each difficulty `steps = 1 … 5` and sample `G = 8` responses for each, 480
responses in total. Per prompt this gives a success count `c ∈ {0, …, 8}`. Per difficulty we
report pass@1 with its standard error over prompts, the fraction of responses that are
**well-formed** (one parseable Answer line), and pass@k from the unbiased estimator.

`Q.rollout` and `Q.response_texts` are the shared sampling helpers (also used in S1–S4):

In [ ]:
print(inspect.getsource(Q.rollout))
print(inspect.getsource(Q.response_texts))

In [ ]:
torch.cuda.reset_peak_memory_stats()
tok = Q.load_tokenizer()
policy = Q.load_chat_model()
MAX_NEW, G, LEVELS, N_PER_LEVEL = 224, 8, [1, 2, 3, 4, 5], 12

def sample_texts(model, problems, n, bs=192):
    """n responses per problem at temperature 1, in problem order; also returns truncation flags."""
    prompts = [s5.prompt_text(p) for p in problems for _ in range(n)]
    texts, truncated = [], []
    for i in range(0, len(prompts), bs):
        ro = Q.rollout(model, tok, prompts[i:i + bs], max_new=MAX_NEW)
        texts += Q.response_texts(tok, ro)
        truncated += ro["truncated"].tolist()
    return texts, torch.tensor(truncated)

def grade(texts, golds):
    """Per response: correct (reward 1) and well-formed (one parseable Answer line)."""
    correct = torch.tensor([verifier_reward(t, g) == 1.0 for t, g in zip(texts, golds)])
    formatted = torch.tensor([verifier_reward(t, g, alpha=1.0) > 0 for t, g in zip(texts, golds)])
    return correct, formatted

print("prompt example:\n" + Q.chat_prompt(tok, s5.prompt_text(s5.make_problems(1, 2, 1)[0])))

In [ ]:
torch.manual_seed(1); t0 = time.time()
measure_set = [p for d in LEVELS for p in s5.make_problems(seed=1, steps=d, n=N_PER_LEVEL)]
m_texts, m_trunc = sample_texts(policy, measure_set, G)
m_correct, m_formatted = grade(m_texts, [p["answer"] for p in measure_set for _ in range(G)])
succ = m_correct.view(-1, G).sum(1)                       # successes per prompt, out of G
level_of = torch.tensor([p["steps"] for p in measure_set])
rows = []
for d in LEVELS:
    sel = level_of == d
    p_prompt = succ[sel].float() / G
    rows.append({"steps": d, "pass@1": p_prompt.mean().item(), "SE": (p_prompt.std() / sel.sum().sqrt()).item(),
                 "well-formed": m_formatted.view(-1, G)[sel].float().mean().item(),
                 "truncated": m_trunc.view(-1, G)[sel].float().mean().item(),
                 "never solved": int((succ[sel] == 0).sum()), "always solved": int((succ[sel] == G).sum()),
                 **{f"pass@{k}": np.mean([pass_at_k(G, int(c), k) for c in succ[sel]]) for k in (2, 4, 8)}})
level_table = pd.DataFrame(rows).set_index("steps")
print(f"{len(m_texts)} responses in {time.time() - t0:.0f}s")
print(level_table.round(3).to_string())
for i in (0, 1):
    print("\n--- example response:", "correct" if m_correct[i] else "not correct", "| gold", measure_set[0]["answer"])
    print(m_texts[i][-300:])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
ax.errorbar(LEVELS, level_table["pass@1"], yerr=level_table["SE"], color=BLUE, marker="o", capsize=3,
            label="pass@1 (± 1 SE over 12 prompts)")
ax.plot(LEVELS, level_table["well-formed"], color=ORANGE, marker="s", label="well-formed (one parseable Answer line)")
ax.plot(LEVELS, level_table["truncated"], color=GRAY, marker="^", label=f"truncated at {MAX_NEW} tokens")
ax.set(xlabel="difficulty: number of operations", ylabel="fraction of responses", ylim=(0, 1),
       title="Qwen3-0.6B, thinking off, temperature 1")
ax.set_xticks(LEVELS); ax.legend(fontsize=8)
ax = axes[1]
shades = plt.cm.Blues(np.linspace(0.4, 1.0, len(LEVELS)))
for j, d in enumerate(LEVELS):
    cs = succ[level_of == d]
    ax.plot(range(1, G + 1), [np.mean([pass_at_k(G, int(c), k) for c in cs]) for k in range(1, G + 1)],
            color=shades[j], marker=".", label=f"steps = {d}")
ax.set(xlabel="k", ylabel="unbiased pass@k", ylim=(0, 1), title=f"pass@k from G = {G} samples per prompt")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Qwen3-0.6B success and well-formed rate per difficulty, and pass@k curves](../figures/chapter-13/study-path-s5_rlvr-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

In the reference run, pass@1 fell from about 0.7 at one operation to about 0.02 at five. At every
level the well-formed rate was well below 1 (0.71 at one operation, 0.27 at five), so a large part of
the zero rewards are format failures under the strict contract, not wrong arithmetic. The model often
writes `Answer: **30**` or `Final answer: Answer: 30`. pass@8 is much higher than pass@1 at the middle
levels: the model can produce the right answer but doesn't do so reliably. RL can exploit exactly this
kind of gap, because it only reinforces answers the policy already samples sometimes.

The table shows the measurement GRPO needs: per prompt, how often the current policy succeeds. A
prompt with `c = 0` of 8 gives an all-wrong group, and `c = 8` gives an all-right group. Both give
zero advantage, so the **training pool** keeps only prompts with `0 < c < G`. This is the filter
used in DAPO-style dynamic sampling and in the difficulty filtering of open RLVR recipes, measured
here before training rather than during it.

### Prediction

## Exercise 7 · select the training pool

`successes` is a long tensor `[P]` of correct counts out of `n`. One line: the boolean mask of
prompts that are solved sometimes.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_select_pool(successes, n):
    return ...                   # True where 0 < successes < n

In [ ]:
ex7_ok = s5.check_select_pool(globals().get("my_select_pool"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def select_pool_ref(successes, n):
    return (successes > 0) & (successes < n)

s5.check_select_pool(select_pool_ref, strict=True)
select_pool = mine_or_ref(globals(), "my_select_pool", ex7_ok, select_pool_ref)
keep = select_pool(succ, G)
train_pool = [p for p, k in zip(measure_set, keep.tolist()) if k]
p_hat_pool = succ[keep].float() / G
print(f"training pool: {len(train_pool)} of {len(measure_set)} prompts "
      f"({int((succ == 0).sum())} never solved, {int((succ == G).sum())} always solved, dropped)")
print("kept per difficulty:", {d: int(keep[level_of == d].sum()) for d in LEVELS})

### Reference explanation

### Reference answer and mechanism

`(successes > 0) & (successes < n)`. The pool keeps prompts that the model solves sometimes, which
are the only ones whose groups can have nonzero advantages. Two caveats. `p̂ = c/8` is noisy: a
prompt with true `p = 0.05` shows `c = 0` two thirds of the time and is dropped, and a kept prompt
can still give constant groups later. And filtering on measured success uses the same policy that
is about to be trained, so the pool changes what "training accuracy" means; held-out problems in
Part F are drawn fresh from the generator with another seed.

---

# Part F · Qwen3: GRPO + verifier, then a held-out evaluation

One iteration of the loop, with the same pieces as S4 and the verifier in place of `rm`:

1. sample 8 prompts from the training pool and `G = 8` responses each (64 rollouts), temperature 1;
2. score every response with `verifier_reward` (α = 0) and compute group advantages `A_i` per prompt;
3. compute `log π_θ(a_t | s_t)` for every response token, and the frozen reference's log-probabilities;
4. take one AdamW step on

$$L = -\frac{1}{T_{\text{tot}}} \sum_{i} \sum_{t \in y_i} A_i \log \pi_\theta(a_t \mid s_t) \; + \; \beta \frac{1}{T_{\text{tot}}} \sum_{i,t} k_3(t),$$

where every token of response `i` shares the sequence advantage `A_i`, `T_tot` is the number of
response tokens in the batch, and `k_3 = e^{d} − d − 1` with `d = log π_ref − log π_θ` is the
nonnegative KL estimate (D35), with β = 0.02. As in Part D, one step per batch makes the ratio 1, so
the clipped surrogate reduces to this loss. The forward and backward passes run in chunks of 8
responses to bound memory.

One precision detail matters at this learning rate. AdamW moves each weight by about `lr = 2e-6`
per step. bf16 keeps 8 significant bits, so a weight near 0.02 is stored with spacing
`2^-13 ≈ 1.2e-4`, and a change of `2e-6` rounds back to the same bf16 number. A bf16 policy
would therefore leave most weights untouched. We train **fp32 master weights** (forward passes under
bf16 autocast) and copy them after each step into a bf16 `sampler` model used for generation.

First, the two lines that make this RLVR: the reward call and the advantage.

### Prediction

## Exercise 8 · rewards and advantages in the training loop

`texts` and `golds` have length `B·G`, ordered prompt by prompt (`G` consecutive responses share a
prompt). Two lines: the reward tensor from `verifier_reward`, then the advantages from
`s5.group_advantages` on the rewards reshaped to `[B, G]`, flattened back to `[B·G]`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_rewards_and_advantages(texts, golds, G):
    R = ...                      # [B·G] float: verifier_reward(text, gold) for each response
    A = ...                      # [B·G]: group advantages, normalised within each prompt's G responses
    return R, A

In [ ]:
ex8_ok = s5.check_rewards_and_advantages(globals().get("my_rewards_and_advantages"), verifier_reward)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def rewards_and_advantages_ref(texts, golds, G):
    R = torch.tensor([verifier_reward(t, g) for t, g in zip(texts, golds)], dtype=torch.float32)
    A = s5.group_advantages(R.view(-1, G)).flatten()
    return R, A

s5.check_rewards_and_advantages(rewards_and_advantages_ref, verifier_reward, strict=True)
rewards_and_advantages = mine_or_ref(globals(), "my_rewards_and_advantages", ex8_ok, rewards_and_advantages_ref)

### Reference explanation

### Reference answer and mechanism

`R = tensor([verifier_reward(t, g) …])` and `A = group_advantages(R.view(-1, G)).flatten()`. The
reshape is the essential part: normalising over the whole batch of 64 would compare a response with
responses to *other* problems, so an easy prompt's correct answers would take advantage from a hard
prompt's wrong ones. Per prompt, the baseline is that prompt's own success rate.

The held-out set: 16 fresh problems per difficulty (80 in total) from seed 2, disjoint from the
measurement set. We sample 4 responses per problem at temperature 1 before and after training, with
the same sampling seed, and compare per problem. Evaluating at temperature 1 measures the
distribution that was trained; greedy decoding would be a different system (Day 23 · 03).

In [ ]:
heldout = [p for d in LEVELS for p in s5.make_problems(seed=2, steps=d, n=16)]
assert not {p["question"] for p in heldout} & {p["question"] for p in measure_set}
N_EVAL = 4

def evaluate(model):
    torch.manual_seed(1234)
    texts, truncated = sample_texts(model, heldout, N_EVAL, bs=160)
    correct, formatted = grade(texts, [p["answer"] for p in heldout for _ in range(N_EVAL)])
    return {"correct": correct.view(-1, N_EVAL).float(), "formatted": formatted.view(-1, N_EVAL).float(),
            "truncated": truncated.view(-1, N_EVAL).float(), "texts": texts}

t0 = time.time()
before = evaluate(policy)
print(f"held-out before training ({time.time() - t0:.0f}s): accuracy {before['correct'].mean():.3f} | "
      f"well-formed {before['formatted'].mean():.3f} | truncated {before['truncated'].mean():.3f}")

### Prediction

## Predict 3 · zero-signal groups at the start and after training

The pool keeps prompts with `0 < c < 8` in the measurement. (a) At the first training iteration,
what fraction of the 8 groups do you expect to have all-equal rewards? Use `zero_signal_prob` with
`p̂ = c/8` of the pool prompts. Compare it with the fraction over *all* 60 measured prompts. (b) After
25 updates, will the fraction be higher or lower than at the start, and why?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
pred_pool = float(zero_signal_prob(p_hat_pool, G).mean())
pred_all = float(zero_signal_prob(succ.float() / G, G).mean())
measured_all = float(((succ == 0) | (succ == G)).float().mean())
print(f"all 60 measured prompts: {measured_all:.2f} of groups were constant (prediction from p̂: {pred_all:.2f})")
print(f"filtered pool: expected constant-group fraction at iteration 1 = {pred_pool:.2f}")

### Reference explanation

### Reference answer and mechanism

(a) In the measurement itself, the constant-group fraction over all 60 prompts is the share with
`c = 0` or `c = 8`. The prediction from `p̂` is for a *fresh* group per prompt and comes out a little
higher, because a prompt measured at `c = 1` or `c = 7` would often give a constant group again. The
filter removes the prompts with `p̂ = 0` or `1`, so the expected fraction in the pool is much lower,
but not zero: a prompt measured at `c = 1` has `p̂ = 1/8`, and `(7/8)^8 + (1/8)^8 ≈ 0.34`, so a third
of its fresh groups are still all-wrong. The printed number is the average over the pool.

(b) It **rises**, for the same reason as in the toy (Part D): as training raises success on the pool
prompts, all-correct groups become frequent. In the reference run the fraction averaged about 0.15
over the first five iterations, close to the prediction, and about 0.85 over the last five, when the
training reward was near 0.97. Each per-iteration value is a multiple of 1/8 (8 groups), so read the
trend in the figure below, not single points.

In [ ]:
w = torch.tensor(0.02, dtype=torch.bfloat16)
print("bf16: 0.02 + 2e-6 == 0.02 →", bool(w + 2e-6 == w), "| fp32:", bool(torch.tensor(0.02) + 2e-6 == torch.tensor(0.02)))
ref_model = Q.load_chat_model().eval().requires_grad_(False)          # frozen π_ref (bf16)
sampler = policy                                                      # bf16 copy, used only for sampling
policy = Q.load_chat_model(dtype=torch.float32)                       # fp32 master weights: the trained policy
print(inspect.getsource(Q.token_logps))
ITERS, PROMPTS_PER_ITER, LR, BETA, CHUNK = 25, 8, 2e-6, 0.02, 8
opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)

def grpo_iteration(problems):
    prompts = [s5.prompt_text(p) for p in problems for _ in range(G)]
    golds = [p["answer"] for p in problems for _ in range(G)]
    ro = Q.rollout(sampler, tok, prompts, max_new=MAX_NEW)              # current weights, bf16 copy
    texts = Q.response_texts(tok, ro)
    R, A = rewards_and_advantages(texts, golds, G)                     # Exercise 8
    A = A.to(DEVICE)
    n_tok, kl_sum = ro["mask"].sum(), 0.0
    for i in range(0, len(prompts), CHUNK):                            # chunks bound the [B, T, V] logits
        sub = {k: (v[i:i + CHUNK] if torch.is_tensor(v) and v.dim() > 0 else v) for k, v in ro.items()}
        mask = sub["mask"]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logp = Q.token_logps(policy, sub)                          # with gradient, fp32 weights
        with torch.no_grad():
            ref_logp = Q.token_logps(ref_model, sub)
        d = ref_logp - logp
        k3 = torch.exp(d) - d - 1
        loss = (-(A[i:i + CHUNK, None] * logp) * mask).sum() / n_tok + BETA * (k3 * mask).sum() / n_tok
        loss.backward()
        kl_sum += (k3 * mask).sum().item()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step(); opt.zero_grad(set_to_none=True)
    with torch.no_grad():                                              # sampler ← updated master weights
        for p_s, p_m in zip(sampler.parameters(), policy.parameters()):
            p_s.copy_(p_m)
    formatted = [verifier_reward(t, g, alpha=1.0) > 0 for t, g in zip(texts, golds)]
    return {"reward": R.mean().item(), "formatted": float(np.mean(formatted)),
            "zero_signal": (R.view(-1, G).std(1) == 0).float().mean().item(),
            "kl": kl_sum / n_tok.item(), "length": ro["mask"].sum(1).mean().item()}

rng = random.Random(3); torch.manual_seed(3)
t0 = time.time(); train_log = []
for it in range(ITERS):
    train_log.append(grpo_iteration(rng.sample(train_pool, PROMPTS_PER_ITER)))
    if it % 5 == 4:
        last = pd.DataFrame(train_log[-5:]).mean()
        print(f"iter {it + 1:2d} | reward {last.reward:.3f} | well-formed {last.formatted:.3f} | "
              f"zero-signal {last.zero_signal:.2f} | KL {last.kl:.4f} | length {last.length:.0f} | {time.time() - t0:.0f}s")
print(f"peak GPU memory so far in Parts E–F: {Q.gib(torch.cuda.max_memory_allocated()):.1f} GiB")

In [ ]:
log = pd.DataFrame(train_log)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
ax = axes[0]
ax.plot(log.index + 1, log.reward, color=BLUE, marker=".", label="mean verifier reward (= accuracy, α = 0)")
ax.plot(log.index + 1, log.formatted, color=ORANGE, marker=".", label="well-formed fraction")
ax.set(xlabel="iteration", ylabel="fraction of 64 rollouts", ylim=(0, 1), title="Training batches")
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(log.index + 1, log.zero_signal, color=BLUE, marker=".", label="measured (8 groups per iteration)")
ax.axhline(pred_pool, color=ORANGE, ls="--", label=f"predicted at start: {pred_pool:.2f}")
ax.set(xlabel="iteration", ylabel="fraction of groups", ylim=(0, 1), title="Groups with zero signal")
ax.legend(fontsize=8)
ax = axes[2]
ax.plot(log.index + 1, log.kl, color=GRAY, marker=".")
ax.set(xlabel="iteration", ylabel="nats per token", title="k3 KL to the reference (before each update)")
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Qwen3 GRPO + verifier: training reward and format, zero-signal fraction, KL to the reference](../figures/chapter-13/study-path-s5_rlvr-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

In [ ]:
del opt, ref_model, policy; Q.free()                                 # training is over; the sampler holds the weights
t0 = time.time()
after = evaluate(sampler)
print(f"held-out after training ({time.time() - t0:.0f}s): accuracy {after['correct'].mean():.3f} | "
      f"well-formed {after['formatted'].mean():.3f} | truncated {after['truncated'].mean():.3f}")
j = next((j for j in range(len(heldout)) if before["formatted"][j].sum() == 0 and after["correct"][j].sum() > 0), 0)
print(f"\nproblem {j} (gold {heldout[j]['answer']}): {heldout[j]['question']}")
print("--- a response before training:\n" + before["texts"][j * N_EVAL][-260:])
k = int(after["correct"][j].argmax())                                # a correct sample after training, if any
print("--- a response after training:\n" + after["texts"][j * N_EVAL + k][-260:])

Is the change larger than the noise? Both evaluations use the same 80 problems, so the right
comparison is **paired**. For problem `j`, let `b_j` and `a_j` be the fraction of its 4 samples
that are correct before and after training, and `d_j = a_j − b_j`. The mean change is `d̄`, and its
standard error is

$$\text{SE}(\bar d) = \frac{\operatorname{std}(d)}{\sqrt{n}}, \qquad n = 80.$$

Pairing removes the variation between problems (some are easy for both models), which the unpaired
SE `sqrt(var(b)/n + var(a)/n)` would count as noise. A change of more than about 2 SE is unlikely
under no real change.

### Prediction

## Exercise 9 · paired standard error of the change

`before` and `after` are float tensors `[n]` of per-problem accuracy. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_paired_se(before, after):
    d = ...                      # per-problem difference
    return float(d.mean()), ...  # mean and its standard error

In [ ]:
ex9_ok = s5.check_paired_se(globals().get("my_paired_se"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def paired_se_ref(before, after):
    d = (after - before).double()
    return float(d.mean()), float(d.std() / math.sqrt(len(d)))

s5.check_paired_se(paired_se_ref, strict=True)
paired_se = mine_or_ref(globals(), "my_paired_se", ex9_ok, paired_se_ref)
changes = {}
for key in ("correct", "formatted", "truncated"):
    m, se = paired_se(before[key].mean(1), after[key].mean(1))
    changes[key] = (m, se)
    print(f"{key:10}: {before[key].mean():.3f} → {after[key].mean():.3f} | change {m:+.3f} ± {se:.3f} (paired SE) "
          f"| z = {m / se if se > 0 else float('nan'):+.1f}")

### Reference explanation

### Reference answer and mechanism

`d = after − before`, then `d.mean()` and `d.std() / sqrt(n)`. The printed `z = mean / SE` says how
many standard errors the change is. Treat `|z| < 2` as "within noise". This SE covers sampling noise
on these 80 problems under one training run; it doesn't cover the variation between training runs
(another seed for the pool order or the rollouts), which is usually larger. One run can't measure
that.

Accuracy is a product of two rates:

$$P(\text{correct}) = P(\text{well-formed}) \times P(\text{correct} \mid \text{well-formed}).$$

The first factor is format compliance (one plain `Answer:` line before the token limit). The second
is the arithmetic, measured only on the responses the verifier could read. Comparing both factors
before and after training separates "learned to write the answer line" from "learned to compute".

### Prediction

## Exercise 10 · split accuracy into format and arithmetic

`formatted` and `correct` are boolean tensors over responses. Two lines: the well-formed rate, and
the correct rate *among well-formed responses only*.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_decompose(formatted, correct):
    p_fmt = ...                  # P(well-formed)
    p_cond = ...                 # P(correct | well-formed): average correct over the formatted responses only
    return float(p_fmt), float(p_cond)

In [ ]:
ex10_ok = s5.check_decompose(globals().get("my_decompose"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def decompose_ref(formatted, correct):
    p_fmt = formatted.float().mean()
    p_cond = correct[formatted].float().mean()
    return float(p_fmt), float(p_cond)

s5.check_decompose(decompose_ref, strict=True)
decompose = mine_or_ref(globals(), "my_decompose", ex10_ok, decompose_ref)

def cond_by_problem(run):
    """Per-problem P(correct | well-formed), NaN where a problem had no well-formed sample."""
    f, c = run["formatted"], run["correct"]
    return torch.where(f.sum(1) > 0, (c * f).sum(1) / f.sum(1).clamp(min=1), torch.tensor(float("nan")))

split = {}
for name, run in (("before", before), ("after", after)):
    split[name] = decompose(run["formatted"].flatten().bool(), run["correct"].flatten().bool())
    print(f"{name:6}: P(well-formed) {split[name][0]:.3f} × P(correct | well-formed) {split[name][1]:.3f} "
          f"= {split[name][0] * split[name][1]:.3f}")
cb, ca = cond_by_problem(before), cond_by_problem(after)
both = ~(cb.isnan() | ca.isnan())
changes["cond"] = paired_se(cb[both], ca[both])
print(f"P(correct | well-formed), paired over {int(both.sum())} problems with a well-formed sample in both runs: "
      f"change {changes['cond'][0]:+.3f} ± {changes['cond'][1]:.3f}")
log_acc = math.log(split["after"][0] * split["after"][1]) - math.log(split["before"][0] * split["before"][1])
share_fmt = (math.log(split["after"][0]) - math.log(split["before"][0])) / log_acc if abs(log_acc) > 1e-9 else float("nan")
print(f"share of the log-accuracy change from format: {share_fmt:.0%} (the rest from P(correct | well-formed))")

### Reference explanation

### Reference answer and mechanism

`formatted.float().mean()` and `correct[formatted].float().mean()`. Dividing by all responses
instead of the well-formed ones would give the accuracy again, not the conditional rate. Because
accuracy is a product, its log is a sum, `log P(correct) = log P(well-formed) + log P(correct | well-formed)`,
so the printed share splits the change in log-accuracy between the two factors.

The paired SE for the conditional rate uses only problems with at least one well-formed sample in
both runs, so its `n` is smaller than 80; read it with that in mind.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.8))
metrics = ["accuracy", "P(well-formed)", "P(correct | well-formed)"]
def per_problem(run):
    return [run["correct"].mean(1), run["formatted"].mean(1), cond_by_problem(run)]
x = np.arange(len(metrics))
for offset, (name, run, color) in zip((-0.18, 0.18), (("before", before, GRAY), ("after", after, BLUE))):
    vals = per_problem(run)
    means = [v[~v.isnan()].mean().item() for v in vals]
    ses = [(v[~v.isnan()].std() / math.sqrt(int((~v.isnan()).sum()))).item() for v in vals]
    bars = ax.bar(x + offset, means, width=0.34, yerr=ses, capsize=3, color=color, label=f"{name} training")
    ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
ax.set_xticks(x, metrics)
ax.set(ylim=(0, 1.1), ylabel="mean over 80 held-out problems",
       title=f"Held-out, 4 samples per problem at T = 1 (± 1 SE over problems); paired accuracy change "
             f"{changes['correct'][0]:+.3f} ± {changes['correct'][1]:.3f}")
ax.title.set_fontsize(9)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
del sampler; Q.free()
print(f"peak GPU memory in Parts E–F: {Q.gib(torch.cuda.max_memory_allocated()):.1f} GiB")

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Held-out accuracy before and after GRPO + verifier, split into format and arithmetic](../figures/chapter-13/study-path-s5_rlvr-06.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

**Reading the result.** In the reference run, held-out accuracy rose from 0.30 to 0.80. The paired
change is +0.50 with a standard error of 0.036, about 14 standard errors, so it is far outside the
sampling noise of these 80 problems. (A development run of the same loop with chunks of 16, which
changes only the order of floating-point sums, reached 0.78.) Most of it is format. The well-formed
rate rose from 0.47 to 0.96, and that factor accounts for about three quarters (74 %) of the change
in log-accuracy. The arithmetic factor rose too: `P(correct | well-formed)` went from 0.65 to 0.84
pooled over responses (the bars average it per problem instead, so they read a little lower), a
paired change of +0.28 ± 0.05 over the 69 problems with a well-formed sample in both runs. Once it
wrote the Answer line, the policy also produced the right number more often. Truncation didn't
change (0.04 → 0.03, z ≈ −0.5), and the mean response length stayed near 90–100 tokens.

What this does and doesn't establish:

- The held-out problems are new numbers, names and operation sequences from the **same generator**
  and difficulty mix as the pool. This is generalisation within one task family, not to other math
  problems or to reasoning in general.
- It is **one training run**. The standard errors cover evaluation sampling on these 80 problems, not
  the variation between runs (pool order, rollouts).
- The format/arithmetic split is an **accounting of a product**, not a causal decomposition. The two
  factors aren't independent: a response with a clean Answer line may also have cleaner working.
- By the last iterations, training reward was near 0.97 and about 85 % of groups gave zero signal. The pool is
  used up, as in the toy run of Part D. Continuing would need a refiltered pool with harder prompts,
  for example `steps = 5` and beyond.

---

# Summary

| Piece | What you built | What was measured |
|---|---|---|
| Extraction (Ex 1) | exactly one `Answer:` line, else `None` | hedged and unmarked responses get no answer |
| Normalisation (Ex 2) | contract regex + exact `Fraction` | `1,042` = 1042, `3.50` = 3.5, `-0` = 0; `1,00` and `**42**` rejected |
| Verifier (Ex 3) | 1 / α / 0 reward | strict: 0 false positives, 3 false negatives on 12 labelled responses; substring: many false positives |
| pass@k (Ex 4) | `1 − C(n−c,k)/C(n,k)` | unbiased in simulation; plug-in is low for k > 1 |
| Zero signal (Ex 5) | `p^G + (1 − p)^G` | predicts the constant-group fraction in the toy run |
| Tabular GRPO (Ex 6) | `−mean(A · log π(a))` | never-solved prompts keep exactly their initial logits; solved prompts stop contributing |
| Pool filter (Ex 7) | `0 < c < G` | Qwen3 prompts solved sometimes, per difficulty |
| RLVR loop (Ex 8) | verifier reward + per-prompt advantages | training reward, format and zero-signal fraction per iteration |
| Evaluation (Ex 9–10) | paired SE; format × arithmetic split | held-out change with its SE and its source |

**Evidence.** The toy results (verifier table, estimator bias, exact zero gradient for never-solved
prompts) are exact or simulated with large samples. The Qwen3 part is one short training run on one
synthetic task family. It verifies the mechanics of RLVR on a real model: a strict verifier, a
difficulty-filtered pool, nonzero advantages and a measured held-out change with a paired standard
error, split into format and arithmetic. It doesn't establish improved reasoning in general, it
doesn't measure the variation between training runs, and any gain that comes from the format factor
is a gain in following the verifier's contract, not in arithmetic.

**Next stage.** [S6 · Combinations: feedback sources × optimizers](s6_combinations.ipynb) treats the
verifier as one feedback source among several (human pairs, a learned reward model, an LLM judge)
and pairs each with an optimizer (selection + SFT, DPO, PPO, GRPO), as real post-training pipelines
do (D32, D40).